In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-14B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


## Qwen-14B-Instruct 

## Qwen-14B-Instruct on Llama 70B

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-14B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████| 8/8 [00:30<00:00,  3.80s/it]


In [8]:
import pandas as pd 
import re


# Load data
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None

# Create results dataframe
results = []

# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["70B_After_Removal"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        query_full = (
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Based on the information provided, select the correct answer choice (A, B, C, D, etc.).\n\n"
            "IMPORTANT: Your response must end with 'Answer: X' where X is the letter of your chosen option.\n"
            "For example: Answer: A"
    )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
        )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
            qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(paths.PREDICTIONS / "Qwen_14B_predictions_progress.csv", index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Qwen_14B_predictions_70B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")


Columns in dataset:
['QA_ID', 'context', 'question_options', 'answer_df3', 'data_source_corr', 'Origin', '70B_sentence_ids', '70B_After_Removal', 'human_sentence_ids', 'Low_Irr_70B', 'Extra_Context_Minus_70B', 'gpt_direct_prediction', '72B_Sentence_Contents', '72B_Low_Irr']
Processing 1297 rows...
Processing row 1/1297...
✅ Processed Merge Q2: Answer = D
Processing row 2/1297...
✅ Processed Merge Q2: Answer = D
Processing row 3/1297...
✅ Processed Merge Q2: Answer = D
Processing row 4/1297...
✅ Processed Merge Q2: Answer = A
Processing row 5/1297...
✅ Processed Merge Q2: Answer = D
Processing row 6/1297...
✅ Processed Merge Q2: Answer = C
Processing row 7/1297...
✅ Processed Merge Q2: Answer = B
Processing row 8/1297...
✅ Processed Merge Q2: Answer = B
Processing row 9/1297...
✅ Processed Merge Q2: Answer = B
Processing row 10/1297...
✅ Processed Merge Q2: Answer = B
Saved progress to CSV after 10 items
Processing row 11/1297...
✅ Processed Merge Q2: Answer = D
Processing row 12/1297..

⚠️ Could not extract answer from response for row 70:
Response: To determine the most likely diagnosis for this patient, let's analyze the key points:

1. **Age**: ...
✅ Processed Merge Q70: Answer = None
Saved progress to CSV after 70 items
Processing row 71/1297...
✅ Processed Merge Q70: Answer = B
Processing row 72/1297...
✅ Processed Merge Q70: Answer = B
Processing row 73/1297...
⚠️ Could not extract answer from response for row 73:
Response: To determine the most likely finding on plain radiographic imaging for this patient, let's analyze t...
✅ Processed Merge Q73: Answer = None
Processing row 74/1297...
⚠️ Could not extract answer from response for row 74:
Response: To determine the most appropriate next step in management for this patient, we need to consider seve...
✅ Processed Merge Q74: Answer = None
Processing row 75/1297...
⚠️ Could not extract answer from response for row 75:
Response: To determine which aspect of the patient's development is abnormal, let's break down t

⚠️ Could not extract answer from response for row 128:
Response: To determine the most likely diagnosis for the patient, let's analyze the key points from the given ...
✅ Processed Merge Q128: Answer = None
Processing row 129/1297...
✅ Processed Merge Q128: Answer = D
Processing row 130/1297...
✅ Processed Merge Q128: Answer = C
Saved progress to CSV after 130 items
Processing row 131/1297...
✅ Processed Merge Q128: Answer = D
Processing row 132/1297...
⚠️ Could not extract answer from response for row 132:
Response: To determine the most likely cause of the patient's symptoms, let's analyze the given details:

- Th...
✅ Processed Merge Q132: Answer = None
Processing row 133/1297...
✅ Processed Merge Q132: Answer = C
Processing row 134/1297...
✅ Processed Merge Q132: Answer = B
Processing row 135/1297...
⚠️ Could not extract answer from response for row 135:
Response: To determine the most appropriate next step in management for this newborn, let's analyze the given ...
✅ Processed Mer

✅ Processed Merge Q180: Answer = B
Processing row 183/1297...
⚠️ Could not extract answer from response for row 183:
Response: To determine the most likely diagnosis for this patient, let's analyze the key points:

- The patien...
✅ Processed Merge Q183: Answer = None
Processing row 184/1297...
⚠️ Could not extract answer from response for row 184:
Response: To determine which factor plays the earliest role in the progression towards the patient's presentat...
✅ Processed Merge Q184: Answer = None
Processing row 185/1297...
⚠️ Could not extract answer from response for row 185:
Response: To determine the most likely cause of the patient's symptoms, let's analyze the given information:

...
✅ Processed Merge Q185: Answer = None
Processing row 186/1297...
✅ Processed Merge Q185: Answer = C
Processing row 187/1297...
⚠️ Could not extract answer from response for row 187:
Response: To determine the most likely condition for this patient, let's analyze the symptoms and signs:

- **...
✅ Pro

⚠️ Could not extract answer from response for row 237:
Response: To determine the most likely etiology of the patient's symptoms, let's analyze the clinical presenta...
✅ Processed Merge Q237: Answer = None
Processing row 238/1297...
⚠️ Could not extract answer from response for row 238:
Response: To determine the most likely diagnosis for this patient, let's analyze the key points:

1. The patie...
✅ Processed Merge Q238: Answer = None
Processing row 239/1297...
✅ Processed Merge Q238: Answer = A
Processing row 240/1297...
⚠️ Could not extract answer from response for row 240:
Response: To determine the appropriate management for this patient, let's analyze the given information step b...
✅ Processed Merge Q240: Answer = None
Saved progress to CSV after 240 items
Processing row 241/1297...
⚠️ Could not extract answer from response for row 241:
Response: To determine the most likely diagnosis for this patient, let's analyze the key points from the clini...
✅ Processed Merge Q241: Answe

...

✅ Processed Merge Q802: Answer = D
Processing row 822/1297...
✅ Processed Merge Q802: Answer = C
Processing row 823/1297...
✅ Processed Merge Q802: Answer = A
Processing row 824/1297...
✅ Processed Merge Q802: Answer = D
Processing row 825/1297...
✅ Processed Merge Q802: Answer = A
Processing row 826/1297...
✅ Processed Merge Q802: Answer = D
Processing row 827/1297...
✅ Processed Merge Q802: Answer = C
Processing row 828/1297...
✅ Processed Merge Q802: Answer = C
Processing row 829/1297...
✅ Processed Merge Q802: Answer = D
Processing row 830/1297...
✅ Processed Merge Q802: Answer = B
Saved progress to CSV after 830 items
Processing row 831/1297...
✅ Processed Merge Q802: Answer = D
Processing row 832/1297...
✅ Processed Merge Q802: Answer = B
Processing row 833/1297...
✅ Processed Merge Q802: Answer = C
Processing row 834/1297...
✅ Processed Merge Q802: Answer = B
Processing row 835/1297...
⚠️ Could not extract answer from response for row 835:
Response: To determine the correct answ

✅ Processed Merge Q901: Answer = C
Saved progress to CSV after 920 items
Processing row 921/1297...
✅ Processed Merge Q901: Answer = C
Processing row 922/1297...
✅ Processed Merge Q901: Answer = D
Processing row 923/1297...
✅ Processed Merge Q901: Answer = B
Processing row 924/1297...
✅ Processed Merge Q901: Answer = C
Processing row 925/1297...
✅ Processed Merge Q901: Answer = B
Processing row 926/1297...
✅ Processed Merge Q901: Answer = D
Processing row 927/1297...
✅ Processed Merge Q901: Answer = D
Processing row 928/1297...
✅ Processed Merge Q901: Answer = B
Processing row 929/1297...
✅ Processed Merge Q901: Answer = B
Processing row 930/1297...
✅ Processed Merge Q901: Answer = C
Saved progress to CSV after 930 items
Processing row 931/1297...
✅ Processed Merge Q901: Answer = A
Processing row 932/1297...
✅ Processed Merge Q901: Answer = C
Processing row 933/1297...
✅ Processed Merge Q901: Answer = C
Processing row 934/1297...
✅ Processed Merge Q901: Answer = C
Processing row 935/12

✅ Processed Merge Q1048: Answer = D
Processing row 1050/1297...
✅ Processed Merge Q1048: Answer = A
Saved progress to CSV after 1050 items
Processing row 1051/1297...
✅ Processed Merge Q1048: Answer = B
Processing row 1052/1297...
✅ Processed Merge Q1048: Answer = A
Processing row 1053/1297...
✅ Processed Merge Q1048: Answer = D
Processing row 1054/1297...
✅ Processed Merge Q1048: Answer = B
Processing row 1055/1297...
✅ Processed Merge Q1048: Answer = C
Processing row 1056/1297...
✅ Processed Merge Q1048: Answer = C
Processing row 1057/1297...
✅ Processed Merge Q1048: Answer = D
Processing row 1058/1297...
✅ Processed Merge Q1048: Answer = A
Processing row 1059/1297...
✅ Processed Merge Q1048: Answer = C
Processing row 1060/1297...
✅ Processed Merge Q1048: Answer = C
Saved progress to CSV after 1060 items
Processing row 1061/1297...
✅ Processed Merge Q1048: Answer = B
Processing row 1062/1297...
✅ Processed Merge Q1048: Answer = B
Processing row 1063/1297...
✅ Processed Merge Q1048: A

✅ Processed Merge Q1126: Answer = B
Processing row 1166/1297...
✅ Processed Merge Q1126: Answer = D
Processing row 1167/1297...
✅ Processed Merge Q1126: Answer = B
Processing row 1168/1297...
✅ Processed Merge Q1126: Answer = D
Processing row 1169/1297...
✅ Processed Merge Q1126: Answer = A
Processing row 1170/1297...
✅ Processed Merge Q1126: Answer = C
Saved progress to CSV after 1170 items
Processing row 1171/1297...
✅ Processed Merge Q1126: Answer = A
Processing row 1172/1297...
✅ Processed Merge Q1126: Answer = C
Processing row 1173/1297...
✅ Processed Merge Q1126: Answer = B
Processing row 1174/1297...
✅ Processed Merge Q1126: Answer = A
Processing row 1175/1297...
✅ Processed Merge Q1126: Answer = D
Processing row 1176/1297...
✅ Processed Merge Q1126: Answer = D
Processing row 1177/1297...
✅ Processed Merge Q1126: Answer = B
Processing row 1178/1297...
✅ Processed Merge Q1126: Answer = C
Processing row 1179/1297...
✅ Processed Merge Q1126: Answer = C
Processing row 1180/1297...
✅

✅ Processed Merge Q1189: Answer = A
Processing row 1284/1297...
✅ Processed Merge Q1189: Answer = D
Processing row 1285/1297...
✅ Processed Merge Q1189: Answer = C
Processing row 1286/1297...
✅ Processed Merge Q1189: Answer = B
Processing row 1287/1297...
✅ Processed Merge Q1189: Answer = B
Processing row 1288/1297...
✅ Processed Merge Q1189: Answer = B
Processing row 1289/1297...
✅ Processed Merge Q1189: Answer = A
Processing row 1290/1297...
✅ Processed Merge Q1189: Answer = B
Saved progress to CSV after 1290 items
Processing row 1291/1297...
✅ Processed Merge Q1189: Answer = B
Processing row 1292/1297...
✅ Processed Merge Q1189: Answer = C
Processing row 1293/1297...
✅ Processed Merge Q1189: Answer = C
Processing row 1294/1297...
✅ Processed Merge Q1189: Answer = C
Processing row 1295/1297...
✅ Processed Merge Q1189: Answer = B
Processing row 1296/1297...
✅ Processed Merge Q1189: Answer = B
Processing row 1297/1297...
✅ Processed Merge Q1189: Answer = B
Saved all predictions to Qwen

In [10]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "Qwen_14B_predictions_70B.csv")
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_df3']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_corr' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_corr' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_corr'] = df['data_source_corr']
else:
    print("Warning: 'data_source_corr' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_corr' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.4711 (47.11%)
Standard Deviation: 0.4994
95% Confidence Interval: [0.4439, 0.4983]
95% CI (percentage): [44.39%, 49.83%]
Sample Size: 1297

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.627148 0.483979 0.020062     0.587746     0.666550        62.714777  48.397925      58.774570      66.654983
      medbullets    207       0.309179 0.463275 0.032200     0.245695     0.372662        30.917874  46.327538      24.569521      37.266228
        medxpert    315       0.114286 0.318664 0.017955     0.078959     0.149612        11.428571  31.866418       7.895900      14.961243
            mmlu    193       0.756477 0.430325 0.030975     0.695381     0.817573        75.647668  43.032452      69.538084      81.757253


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.4711 (47.11%)